# 🚑 Notebook 1 — Emergency Severity Classifier

**Project:** AI-Enabled Smart Emergency Response & Ambulance Coordination System
**Model role:** Triage incoming emergency calls into 5 severity levels so the dispatch engine can prioritise critical cases.

| Severity | Label          | Examples                                |
|----------|----------------|-----------------------------------------|
| 1        | Critical       | Cardiac arrest, unconscious, severe trauma |
| 2        | Serious        | Stroke symptoms, major bleeding         |
| 3        | Moderate       | Fractures, abdominal pain               |
| 4        | Minor          | Cuts, sprains, mild fever               |
| 5        | Non-Emergency  | Headache, advisory only                 |

### 🎯 Performance Target
- **Accuracy ≥ 0.95**
- **Macro F1 ≥ 0.93** (matters more than accuracy because Critical class is rare)
- **Per-class recall on Critical class ≥ 0.95** (missing a Critical is unacceptable)

### 🧰 What this notebook does
1. Generates 50 000 realistic synthetic emergency records
2. Performs deep EDA on vitals, symptoms and demographics
3. Trains 5 models, picks the best, ensembles the top performers
4. Calibrates probabilities (so confidence scores are reliable for the dispatcher UI)
5. Saves the production-ready artefacts to `models/`

## 1 · Setup & Imports

In [ ]:
# If anything is missing, uncomment:
# !pip install -q numpy pandas scikit-learn xgboost lightgbm catboost imbalanced-learn shap matplotlib seaborn joblib

import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve, auc)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("Setup complete ✔")

## 2 · Synthetic Data Generation

We build a **rule-driven generator** that mimics how a senior triage nurse would label cases.
The rules use vitals (Glasgow Coma Scale, SpO₂, pulse, BP, respiratory rate), age, and
the symptom mix. We deliberately add ~5 % label noise so the model has to *learn* and not
memorise — but the underlying signal is strong enough for >95 % accuracy.

In [ ]:
SYMPTOMS_POOL = [
    # Critical-tier symptoms
    "cardiac_arrest", "unconscious", "severe_burns", "spinal_injury",
    "anaphylaxis", "major_bleeding",
    # Serious-tier
    "stroke_symptoms", "chest_pain", "shortness_of_breath", "seizure",
    "head_trauma", "diabetic_emergency",
    # Moderate-tier
    "fracture", "moderate_bleeding", "abdominal_pain", "high_fever",
    # Minor / Non-emergency tier
    "vomiting", "dizziness", "minor_cut", "sprain", "headache",
]
N_SAMPLES = 50_000


def _vitals_for_severity(sev: int):
    """Sample vitals from severity-conditional distributions."""
    if sev == 1:                            # Critical
        gcs   = np.clip(np.random.normal(6, 2), 3, 15)
        spo2  = np.clip(np.random.normal(82, 6), 60, 100)
        pulse = np.clip(np.random.normal(140, 25), 30, 220)
        rr    = np.clip(np.random.normal(32, 6), 8, 50)
        sbp   = np.clip(np.random.normal(80, 20), 50, 200)
    elif sev == 2:                          # Serious
        gcs   = np.clip(np.random.normal(11, 2), 3, 15)
        spo2  = np.clip(np.random.normal(91, 4), 70, 100)
        pulse = np.clip(np.random.normal(115, 18), 50, 200)
        rr    = np.clip(np.random.normal(24, 4), 10, 45)
        sbp   = np.clip(np.random.normal(105, 18), 60, 200)
    elif sev == 3:                          # Moderate
        gcs   = np.clip(np.random.normal(14, 1), 10, 15)
        spo2  = np.clip(np.random.normal(95, 3), 85, 100)
        pulse = np.clip(np.random.normal(95, 14), 55, 160)
        rr    = np.clip(np.random.normal(20, 3), 12, 35)
        sbp   = np.clip(np.random.normal(125, 12), 80, 180)
    elif sev == 4:                          # Minor
        gcs   = 15
        spo2  = np.clip(np.random.normal(97, 2), 92, 100)
        pulse = np.clip(np.random.normal(85, 10), 55, 130)
        rr    = np.clip(np.random.normal(17, 2), 12, 25)
        sbp   = np.clip(np.random.normal(125, 10), 95, 160)
    else:                                   # Non-emergency
        gcs   = 15
        spo2  = np.clip(np.random.normal(98, 1), 95, 100)
        pulse = np.clip(np.random.normal(78, 8), 55, 110)
        rr    = np.clip(np.random.normal(16, 2), 12, 22)
        sbp   = np.clip(np.random.normal(122, 8), 100, 150)
    return gcs, spo2, pulse, rr, sbp


def _symptoms_for_severity(sev: int):
    """Pick symptom set whose tier matches the severity."""
    if sev == 1:
        primary = np.random.choice(SYMPTOMS_POOL[:6], size=np.random.randint(1, 3),
                                   replace=False).tolist()
        extras  = np.random.choice(SYMPTOMS_POOL[6:],  size=np.random.randint(0, 2),
                                   replace=False).tolist()
    elif sev == 2:
        primary = np.random.choice(SYMPTOMS_POOL[6:12], size=np.random.randint(1, 3),
                                   replace=False).tolist()
        extras  = np.random.choice(SYMPTOMS_POOL[12:], size=np.random.randint(0, 2),
                                   replace=False).tolist()
    elif sev == 3:
        primary = np.random.choice(SYMPTOMS_POOL[12:16], size=np.random.randint(1, 3),
                                   replace=False).tolist()
        extras  = np.random.choice(SYMPTOMS_POOL[16:], size=np.random.randint(0, 2),
                                   replace=False).tolist()
    elif sev == 4:
        primary = np.random.choice(SYMPTOMS_POOL[16:],  size=np.random.randint(1, 3),
                                   replace=False).tolist()
        extras  = []
    else:
        primary = np.random.choice(SYMPTOMS_POOL[-3:], size=1, replace=False).tolist()
        extras  = []
    return list(set(primary + extras))


def generate_severity_dataset(n=N_SAMPLES, label_noise=0.05):
    # Severity prior — Critical is rare in real life
    severity_probs = [0.06, 0.14, 0.30, 0.30, 0.20]
    rows = []
    for _ in range(n):
        true_sev = np.random.choice([1, 2, 3, 4, 5], p=severity_probs)
        gcs, spo2, pulse, rr, sbp = _vitals_for_severity(true_sev)
        dbp = max(40, sbp - np.random.randint(30, 55))
        symptoms = _symptoms_for_severity(true_sev)
        age      = int(np.clip(np.random.normal(45, 22), 1, 100))
        gender   = np.random.choice([0, 1])    # 0=female, 1=male

        # Inject label noise (mis-triages happen in real life)
        observed_sev = true_sev
        if np.random.rand() < label_noise:
            observed_sev = np.clip(true_sev + np.random.choice([-1, 1]), 1, 5)

        row = {
            "age": age, "gender": gender,
            "gcs": round(gcs, 1), "spo2": round(spo2, 1),
            "pulse": int(pulse), "resp_rate": int(rr),
            "bp_systolic": int(sbp), "bp_diastolic": int(dbp),
        }
        for s in SYMPTOMS_POOL:
            row[f"sym_{s}"] = int(s in symptoms)
        row["severity"] = observed_sev
        rows.append(row)
    return pd.DataFrame(rows)


df = generate_severity_dataset()
print(f"Generated {len(df):,} rows × {df.shape[1]} columns")
df.head()

## 3 · Exploratory Data Analysis

### 3.1 · Severity class distribution

In [ ]:
class_counts = df["severity"].value_counts().sort_index()
class_labels = {1:"Critical", 2:"Serious", 3:"Moderate", 4:"Minor", 5:"Non-Emergency"}

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar([class_labels[i] for i in class_counts.index],
              class_counts.values,
              color=["#c0392b", "#e67e22", "#f1c40f", "#27ae60", "#3498db"])
for b, v in zip(bars, class_counts.values):
    ax.text(b.get_x()+b.get_width()/2, v+200, f"{v:,}",
            ha="center", fontsize=10, fontweight="bold")
ax.set_title("Severity class distribution (target imbalance check)", fontsize=13)
ax.set_ylabel("Number of cases")
plt.tight_layout(); plt.show()

print("Class imbalance ratio (max / min):",
      round(class_counts.max() / class_counts.min(), 2))

> **Insight:** Critical cases are ~5× rarer than Moderate. We *must* handle this imbalance — SMOTE is applied later.

### 3.2 · Vital signs distribution by severity

In [ ]:
vital_cols = ["gcs", "spo2", "pulse", "resp_rate", "bp_systolic"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
palette = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60", "#3498db"]
for ax, col in zip(axes, vital_cols):
    sns.boxplot(x="severity", y=col, data=df, ax=ax, palette=palette)
    ax.set_title(col.upper())
    ax.set_xlabel("Severity")
plt.suptitle("Vital signs separate cleanly by severity → strong learnable signal",
             fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

### 3.3 · Symptom co-occurrence with severity

In [ ]:
sym_cols = [c for c in df.columns if c.startswith("sym_")]
heat = df.groupby("severity")[sym_cols].mean().T
heat.columns = [class_labels[c] for c in heat.columns]
heat.index = [c.replace("sym_", "") for c in heat.index]

fig, ax = plt.subplots(figsize=(9, 9))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn_r", ax=ax, cbar_kws={"label":"prevalence"})
ax.set_title("Symptom prevalence by severity class", fontsize=13)
plt.tight_layout(); plt.show()

### 3.4 · Age distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.violinplot(x="severity", y="age", data=df, palette=palette, ax=ax, inner="quartile")
ax.set_title("Patient age vs severity")
ax.set_xticklabels([class_labels[i] for i in sorted(df['severity'].unique())])
plt.tight_layout(); plt.show()

### 3.5 · Correlation heatmap (numeric features)

In [ ]:
num_cols = ["age","gender","gcs","spo2","pulse","resp_rate",
            "bp_systolic","bp_diastolic","severity"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax)
ax.set_title("Pearson correlation — numeric features")
plt.tight_layout(); plt.show()

## 4 · Preprocessing
- Stratified train/test split (80/20)
- StandardScaler on continuous features (helps the LR baseline; tree models are scale-invariant but it doesn't hurt)
- SMOTE on the training set only (never touch the test set)

In [ ]:
TARGET = "severity"
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols].values
y = df[TARGET].values - 1            # 0-indexed for XGBoost

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  Test: {X_test_s.shape}")
print("Pre-SMOTE class counts:", dict(zip(*np.unique(y_train, return_counts=True))))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_s, y_train)
print("Post-SMOTE class counts:", dict(zip(*np.unique(y_train_bal, return_counts=True))))

## 5 · Model Training & Comparison

We benchmark **5 candidates**, then build a **soft-voting ensemble** of the top boosters
and finally calibrate it so the dispatcher UI gets reliable probability scores.

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, cv=False):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)
    f1  = f1_score(y_te, preds, average="macro")
    cv_acc = None
    if cv:
        cv_acc = cross_val_score(model, X_tr, y_tr,
                                 cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                 scoring="accuracy", n_jobs=-1).mean()
    results[name] = {"model": model, "accuracy": acc, "macro_f1": f1, "cv_accuracy": cv_acc}
    print(f"{name:25s}  acc={acc:.4f}  macro-F1={f1:.4f}"
          + (f"  cv-acc={cv_acc:.4f}" if cv_acc else ""))
    return model

### 5.1 · Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=2000,
                        n_jobs=-1, random_state=RANDOM_STATE)
evaluate("LogisticRegression", lr, X_train_bal, y_train_bal, X_test_s, y_test)

### 5.2 · Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400, max_depth=18, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced")
evaluate("RandomForest", rf, X_train_bal, y_train_bal, X_test_s, y_test)

### 5.3 · XGBoost (tuned)

In [ ]:
xgb = XGBClassifier(
    n_estimators=600, max_depth=8, learning_rate=0.06,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=2, reg_alpha=0.1, reg_lambda=1.2,
    objective="multi:softprob", num_class=5,
    eval_metric="mlogloss", tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1)
evaluate("XGBoost", xgb, X_train_bal, y_train_bal, X_test_s, y_test, cv=True)

### 5.4 · LightGBM (tuned)

In [ ]:
lgb = LGBMClassifier(
    n_estimators=600, max_depth=-1, num_leaves=63,
    learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=0.5,
    objective="multiclass", num_class=5, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
evaluate("LightGBM", lgb, X_train_bal, y_train_bal, X_test_s, y_test, cv=True)

### 5.5 · CatBoost

In [ ]:
cat = CatBoostClassifier(
    iterations=600, depth=8, learning_rate=0.06,
    l2_leaf_reg=3.0, loss_function="MultiClass",
    random_seed=RANDOM_STATE, verbose=False)
evaluate("CatBoost", cat, X_train_bal, y_train_bal, X_test_s, y_test, cv=True)

### 5.6 · Soft-voting ensemble of top boosters

In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ("xgb", XGBClassifier(**xgb.get_params())),
        ("lgb", LGBMClassifier(**lgb.get_params())),
        ("cat", CatBoostClassifier(iterations=600, depth=8, learning_rate=0.06,
                                   l2_leaf_reg=3.0, loss_function="MultiClass",
                                   random_seed=RANDOM_STATE, verbose=False)),
    ],
    voting="soft", n_jobs=-1)
evaluate("VotingEnsemble", ensemble, X_train_bal, y_train_bal, X_test_s, y_test)

### 5.7 · Calibrated ensemble (probability-reliable for the UI)

In [ ]:
calibrated = CalibratedClassifierCV(
    estimator=VotingClassifier(
        estimators=[
            ("xgb", XGBClassifier(**xgb.get_params())),
            ("lgb", LGBMClassifier(**lgb.get_params())),
        ], voting="soft"),
    method="isotonic", cv=3)
evaluate("CalibratedEnsemble", calibrated, X_train_bal, y_train_bal, X_test_s, y_test)

## 6 · Model leaderboard

In [ ]:
leaderboard = (pd.DataFrame([{ "model": k, **{m:v for m,v in v.items() if m!='model'} }
                             for k,v in results.items()])
                 .sort_values("macro_f1", ascending=False)
                 .reset_index(drop=True))
display(leaderboard)

best_name = leaderboard.iloc[0]["model"]
best_model = results[best_name]["model"]
print(f"\n🏆 Best model: {best_name}  "
      f"(acc={results[best_name]['accuracy']:.4f}, F1={results[best_name]['macro_f1']:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = leaderboard.sort_values("macro_f1")
ax.barh(order["model"], order["macro_f1"], color="#2c3e50")
ax.barh(order["model"], order["accuracy"], color="#3498db", alpha=0.55)
ax.axvline(0.95, ls="--", color="red", label="95% target")
ax.set_xlim(0.6, 1.0)
ax.set_xlabel("Score")
ax.set_title("Macro-F1 (dark) vs Accuracy (light) — all models")
ax.legend()
plt.tight_layout(); plt.show()

## 7 · Deep Dive on the Winning Model

### 7.1 · Confusion matrix

In [ ]:
y_pred = best_model.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)
labels = [class_labels[i+1] for i in range(5)]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=labels, yticklabels=labels)
ax.set_title(f"{best_name} — confusion matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

### 7.2 · Per-class precision / recall / F1

In [ ]:
report = classification_report(y_test, y_pred, target_names=labels,
                               digits=4, output_dict=True)
report_df = pd.DataFrame(report).T.round(4)
display(report_df)

per_class = report_df.loc[labels, ["precision","recall","f1-score"]]
fig, ax = plt.subplots(figsize=(10, 4.5))
per_class.plot.bar(ax=ax, color=["#34495e","#e74c3c","#27ae60"])
ax.axhline(0.95, ls="--", color="black", alpha=0.4, label="95% target")
ax.set_title("Per-class metrics — Critical recall is the safety-critical one")
ax.set_ylim(0.5, 1.02); ax.legend(); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

### 7.3 · ROC curves (one-vs-rest)

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0,1,2,3,4])
try:
    y_proba = best_model.predict_proba(X_test_s)
except AttributeError:
    y_proba = None

if y_proba is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, lab in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_test_bin[:,i], y_proba[:,i])
        ax.plot(fpr, tpr, label=f"{lab} (AUC={auc(fpr,tpr):.3f})")
    ax.plot([0,1],[0,1], "k--", alpha=0.4)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC — one-vs-rest"); ax.legend(loc="lower right")
    plt.tight_layout(); plt.show()
    print("Macro AUC:", round(roc_auc_score(y_test_bin, y_proba, multi_class='ovr'), 4))

### 7.4 · Feature importance

In [ ]:
# Pull importances from the underlying XGB inside the ensemble (or model itself)
imp_model = None
if hasattr(best_model, "feature_importances_"):
    imp_model = best_model
elif hasattr(best_model, "estimators_"):
    for est in best_model.estimators_:
        if hasattr(est, "feature_importances_"):
            imp_model = est; break
elif hasattr(best_model, "named_estimators_"):
    imp_model = list(best_model.named_estimators_.values())[0]

if imp_model is not None and hasattr(imp_model, "feature_importances_"):
    imp = pd.Series(imp_model.feature_importances_, index=feature_cols)
    top = imp.sort_values(ascending=True).tail(20)
    fig, ax = plt.subplots(figsize=(9, 7))
    top.plot.barh(ax=ax, color="#16a085")
    ax.set_title("Top-20 feature importances")
    plt.tight_layout(); plt.show()

### 7.5 · SHAP — explain individual predictions

In [ ]:
import shap
try:
    # Pick a tree model from inside the best estimator for a TreeExplainer
    tree_model = None
    for cand in [best_model] + list(getattr(best_model, "estimators_", [])):
        if isinstance(cand, (XGBClassifier, LGBMClassifier)):
            tree_model = cand; break
    if tree_model is None:
        tree_model = xgb           # fall back to standalone XGB

    sample = X_test_s[:500]
    explainer = shap.TreeExplainer(tree_model)
    shap_vals = explainer.shap_values(sample)
    shap.summary_plot(shap_vals, sample, feature_names=feature_cols,
                      plot_type="bar", show=True)
except Exception as e:
    print("SHAP skipped:", e)

## 8 · Sanity check on a fresh emergency

In [ ]:
def triage(case: dict, model=best_model, scaler=scaler):
    row = {c: 0 for c in feature_cols}
    row.update(case)
    x = scaler.transform([[row[c] for c in feature_cols]])
    pred = int(model.predict(x)[0]) + 1
    proba = model.predict_proba(x)[0] if hasattr(model, "predict_proba") else None
    return {"severity": pred, "label": class_labels[pred],
            "confidence": float(proba.max()) if proba is not None else None}

print("65 yo male, cardiac arrest, GCS 6:")
print(triage({"age":65,"gender":1,"gcs":6,"spo2":78,"pulse":160,"resp_rate":34,
              "bp_systolic":75,"bp_diastolic":50,"sym_cardiac_arrest":1,
              "sym_chest_pain":1,"sym_unconscious":1}))

print("\n28 yo female, sprained ankle:")
print(triage({"age":28,"gender":0,"gcs":15,"spo2":98,"pulse":80,"resp_rate":16,
              "bp_systolic":120,"bp_diastolic":78,"sym_sprain":1}))

print("\n40 yo, mild headache:")
print(triage({"age":40,"gender":0,"gcs":15,"spo2":99,"pulse":72,"resp_rate":15,
              "bp_systolic":118,"bp_diastolic":76,"sym_headache":1}))

## 9 · Persist the artefacts

In [ ]:
joblib.dump(best_model,  "models/severity_classifier.pkl")
joblib.dump(scaler,      "models/severity_scaler.pkl")
joblib.dump(feature_cols,"models/severity_feature_cols.pkl")

final_metrics = {
    "best_model": best_name,
    "accuracy":   round(results[best_name]["accuracy"], 4),
    "macro_f1":   round(results[best_name]["macro_f1"], 4),
    "per_class_recall": {labels[i]: round(report[labels[i]]["recall"], 4) for i in range(5)},
}
with open("reports/severity_classifier_report.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps(final_metrics, indent=2))
print("\n✅ Severity classifier ready for the dispatch backend.")

## 10 · Summary

| Metric              | Target | Achieved |
|---------------------|--------|----------|
| Accuracy            | ≥ 0.95 | _see leaderboard above_ |
| Macro F1            | ≥ 0.93 | _see leaderboard above_ |
| Critical-class recall | ≥ 0.95 | _see per-class chart_ |

**What we did right**
- Severity-conditional vital-sign sampling → strong, learnable signal
- SMOTE → no ignored Critical class
- 5-model bake-off → best architecture for the data
- Calibrated probabilities → reliable confidence % for the dispatcher UI
- SHAP → explainable per-case decisions

→ Continue to **Notebook 2 — ETA Predictor**.